# 🥉 Bronze Layer — Raw Ingestion

> **Purpose**: Land raw data from source systems into Delta Lake with zero transformation.
> Preserves full audit trail with `_ingested_at`, `_batch_id`, and `_source_file` columns.

## Pipeline
```
Azure SQL / ADLS / REST API  →  ADF Copy  →  Parquet landing  →  Delta MERGE (Bronze)
```

In [ ]:
# ── Parameters (override via ADF / Fabric Pipeline) ───────────────────────
dbutils.widgets.text('source_table',   'dbo.sales',        'Source Table')
dbutils.widgets.text('load_type',      'incremental',      'Load Type: full | incremental')
dbutils.widgets.text('batch_id',       '',                 'ADF Pipeline Run ID')
dbutils.widgets.text('storage_account','<storage_account>','ADLS Gen2 Account Name')
dbutils.widgets.text('lakehouse_path', 'Files/bronze',     'Lakehouse Bronze Root Path')

source_table    = dbutils.widgets.get('source_table')
load_type       = dbutils.widgets.get('load_type')
batch_id        = dbutils.widgets.get('batch_id') or spark.sparkContext.applicationId
storage_account = dbutils.widgets.get('storage_account')
lakehouse_path  = dbutils.widgets.get('lakehouse_path')

print(f'Source     : {source_table}')
print(f'Load type  : {load_type}')
print(f'Batch ID   : {batch_id}')

In [ ]:
# ── Spark Configuration ────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from delta.tables import DeltaTable
import re, datetime

spark.conf.set('spark.sql.shuffle.partitions', '8')
spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')
spark.conf.set('spark.databricks.delta.autoCompact.enabled', 'true')

# Derive safe table name and Delta path
table_safe   = re.sub(r'[^a-zA-Z0-9_]', '_', source_table).strip('_')
delta_path   = f'abfss://bronze@{storage_account}.dfs.core.windows.net/{table_safe}'
print(f'Delta path : {delta_path}')

In [ ]:
# ── Read Parquet Landing Zone (output from ADF Copy) ──────────────────────
landing_path = f'abfss://landing@{storage_account}.dfs.core.windows.net/{table_safe}/batch={batch_id}'

df_raw = (
    spark.read
    .option('recursiveFileLookup', 'true')
    .parquet(landing_path)
)

print(f'Rows read from landing: {df_raw.count():,}')
df_raw.printSchema()

In [ ]:
# ── Add Audit Columns (zero business transformation) ──────────────────────
df_bronze = (
    df_raw
    .withColumn('_ingested_at',  F.current_timestamp().cast(TimestampType()))
    .withColumn('_batch_id',     F.lit(batch_id))
    .withColumn('_source_table', F.lit(source_table))
    .withColumn('_source_file',  F.input_file_name())
    .withColumn('_load_type',    F.lit(load_type))
)

print('Schema after audit columns:')
df_bronze.printSchema()

In [ ]:
# ── Write to Delta Lake (MERGE for incremental, OVERWRITE for full) ───────
if load_type == 'full':
    # Full load — truncate and reload
    (
        df_bronze.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .option('path', delta_path)
        .saveAsTable(f'bronze.{table_safe}')
    )
    print(f'Full load complete: {df_bronze.count():,} rows written')

else:
    # Incremental — MERGE on primary key to avoid duplicates
    pk_col = '_batch_id'   # Adjust to actual PK column for your table

    if DeltaTable.isDeltaTable(spark, delta_path):
        dt = DeltaTable.forPath(spark, delta_path)
        (
            dt.alias('target')
            .merge(
                df_bronze.alias('source'),
                f'target.{pk_col} = source.{pk_col}'
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print('Incremental MERGE complete')
    else:
        # First load — create Delta table
        (
            df_bronze.write
            .format('delta')
            .mode('overwrite')
            .option('path', delta_path)
            .saveAsTable(f'bronze.{table_safe}')
        )
        print(f'Initial load complete: {df_bronze.count():,} rows')

In [ ]:
# ── Optimize Delta Table ───────────────────────────────────────────────────
spark.sql(f'OPTIMIZE bronze.{table_safe}')
print('OPTIMIZE complete')

# ── Row count validation ───────────────────────────────────────────────────
final_count = spark.table(f'bronze.{table_safe}').count()
print(f'Bronze table row count: {final_count:,}')
mssparkutils.notebook.exit(str(final_count))